# BIOT 6900 · Module 2 — Multi-Omics Target Identification & Validation
### Week 2 · Computational Lab 2 · Starter Notebook

**What this notebook does.** You'll integrate three data layers — transcriptomics (RNA), proteomics (protein), and genomics — to nominate and *rank* disease targets, then hand that ranked list to Weeks 3–4.

**How today is built — I do → we do → you do:**
- **Part 1 (worked, already complete):** CPTAC **breast cancer** on *synthetic* data. The tumors are measured at all three layers, so the data is *sample-matched* — you integrate **at the sample level**. Your instructor runs these cells; follow along.
- **Part 2 (together, in class):** find and load *real* CPTAC data, then re-run Part 1 on real numbers. Not graded.
- **Part 3 (your assignment, `# TODO` cells):** **Alzheimer's disease**. The cohorts are *not* matched across layers, so you integrate **at the gene level**. You transfer the Part 1 method to messier, more realistic data.

> **The one idea to carry through:** *how* you integrate is dictated by *what your data has matched.* Matched → correlate within samples. Unmatched → compare gene-level summaries. Same goal, different machinery.

**Data.** Part 1 looks for the synthetic CPTAC files in `data/` and, if they're absent, falls back to clearly-labelled demo data so it always runs. Part 3 requires the three Alzheimer's matrices posted on **Canvas** — place them in `data/` (see the Lab Guide).

*Continuity from Module 1:* you'll see **TP53 / p53** (`P04637`, PDB `1TUP`) surface in the cancer half and **APOE** (`rs7412`) in the Alzheimer's half.

**Before you submit:** `Kernel → Restart & Run All` so the whole notebook executes top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

RNG = np.random.default_rng(6900)  # fixed seed so results are reproducible

# Module 1 continuity anchors
P53_UNIPROT, P53_PDB = "P04637", "1TUP"   # p53  -> cancer half
APOE_VARIANT = "rs7412"                    # APOE -> Alzheimer's half

# Default scoring weights for this course: equal across the three layers.
# You MAY change these in Part 3 — but if you do, you must justify it in your report.
EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}

## Helper functions you'll use in both parts
Two small functions do the scoring work. `rank_percentile` puts any layer's scores on a common 0–1 scale (by rank, so it's robust to outliers and to layers being on different units). `multi_evidence_score` normalizes each layer and takes the weighted sum.

In [2]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## Data loading (provided — you don't need to edit these)
`load_cptac()` reads the matched breast-cancer matrices for Part 1 (or synthesizes demo data if they're missing). `load_ad()` reads the three Alzheimer's matrices you download from Canvas for Part 2.

In [3]:
CPTAC_GENES = ["TP53", "PIK3CA", "ERBB2", "ESR1", "GATA3", "MYC", "CDH1",
               "MAP3K1", "PTEN", "AKT1", "RB1", "CCND1", "FOXA1", "MKI67",
               "EGFR", "BRCA1", "BRCA2", "KRT5", "VIM", "ACTB"]


def _synth_cptac(n_tumor=60, n_normal=15):
    genes = CPTAC_GENES
    tumor_ids = [f"T{i:02d}" for i in range(n_tumor)]
    normal_ids = [f"N{i:02d}" for i in range(n_normal)]
    coupling = pd.Series(RNG.uniform(0.05, 0.80, len(genes)), index=genes)

    def gene_matrix(sample_ids, shift):
        rna = pd.DataFrame(RNG.normal(0, 1, (len(genes), len(sample_ids))),
                           index=genes, columns=sample_ids).add(shift, axis=0)
        noise = pd.DataFrame(RNG.normal(0, 1, rna.shape), index=genes, columns=sample_ids)
        prot = rna.mul(coupling, axis=0) + noise.mul(1 - coupling + 0.35, axis=0)
        return rna, prot

    shift = pd.Series(0.0, index=genes)
    for g in ["TP53", "ERBB2", "MKI67", "MYC", "PIK3CA"]:
        shift[g] = RNG.uniform(1.2, 2.2)
    rna_t, prot_t = gene_matrix(tumor_ids, shift)
    rna_n, prot_n = gene_matrix(normal_ids, pd.Series(0.0, index=genes))
    rna = pd.concat([rna_t, rna_n], axis=1)
    prot = pd.concat([prot_t, prot_n], axis=1)
    mut_freq = pd.Series(RNG.uniform(0.0, 0.05, len(genes)), index=genes)
    for g in ["TP53", "PIK3CA", "CDH1", "GATA3"]:
        mut_freq[g] = RNG.uniform(0.25, 0.45)
    meta = pd.Series(["tumor"] * n_tumor + ["normal"] * n_normal,
                     index=tumor_ids + normal_ids, name="group")
    return rna, prot, mut_freq, meta


def load_cptac():
    p = {"rna": "data/cptac_brca_rna.tsv", "prot": "data/cptac_brca_protein.tsv",
         "mut": "data/cptac_brca_mutation.tsv"}
    if all(os.path.exists(v) for v in p.values()):
        rna = pd.read_csv(p["rna"], sep="\t", index_col=0)
        prot = pd.read_csv(p["prot"], sep="\t", index_col=0)
        mut = pd.read_csv(p["mut"], sep="\t", index_col=0).iloc[:, 0]
        print("Loaded CPTAC files from data/.")
        return rna, prot, mut, None
    print("!! WARNING: CPTAC files not found -> SYNTHETIC demo data "
          "(illustrative only, not real CPTAC values).")
    return _synth_cptac()


def load_ad():
    p = {"tx": "data/ad_transcriptomics.tsv", "pr": "data/ad_proteomics.tsv",
         "gw": "data/ad_gwas.tsv"}
    missing = [v for v in p.values() if not os.path.exists(v)]
    if missing:
        raise FileNotFoundError(
            "Alzheimer's matrices not found: " + ", ".join(missing) +
            "\nDownload the three files from Canvas and put them in a 'data/' folder "
            "next to this notebook (see the Lab Guide).")
    return (pd.read_csv(p["tx"], sep="\t"),
            pd.read_csv(p["pr"], sep="\t"),
            pd.read_csv(p["gw"], sep="\t"))

---
# Part 1 — Worked example: CPTAC breast cancer *(run and read; nothing to edit)*

**Why integrate at all?** Each layer can produce artifacts the others don't share, so a target supported across genome, transcriptome, and proteome is a stronger bet than one seen in only one layer. Integration is triangulation.

Because CPTAC measures the **same tumors** at every layer, we can line samples up and integrate **at the sample level.**

In [4]:
rna, prot, mut_freq, meta = load_cptac()
print("RNA matrix:    ", rna.shape, "(genes x samples)")
print("Protein matrix:", prot.shape)
print("Mutation freq: ", mut_freq.shape)

Loaded CPTAC files from data/.
RNA matrix:     (23121, 122) (genes x samples)
Protein matrix: (12621, 122)
Mutation freq:  (9448,)


### 1.2 Harmonize identifiers → a common gene key
The real technical wall in multi-omics: gene symbols, Ensembl IDs, and UniProt accessions don't line up for free. Here we intersect on a shared key and check how many genes *survive the join* — always your first reality check.

In [5]:
common = rna.index.intersection(prot.index).intersection(mut_freq.index)
print(f"Genes surviving the 3-way join: {len(common)} "
      f"(RNA {len(rna.index)}, protein {len(prot.index)}, mutation {len(mut_freq.index)})")
rna, prot, mut_freq = rna.loc[common], prot.loc[common], mut_freq.loc[common]
samples = rna.columns.intersection(prot.columns)

Genes surviving the 3-way join: 6306 (RNA 23121, protein 12621, mutation 9448)


### 1.3–1.4 Sample-level RNA–protein correlation
With matched samples we can ask, per gene, *does protein track RNA across patients?* The answer is usually **partial** — correlation is modest and varies by gene. A gene where protein does **not** track RNA isn't broken data; it's a signal of post-transcriptional regulation (translational buffering, protein turnover).

In [6]:
corr = pd.Series(
    {g: stats.spearmanr(rna.loc[g, samples], prot.loc[g, samples]).statistic
     for g in common}, name="rna_prot_corr")
print(f"RNA-protein correlation: median={corr.median():.2f}, "
      f"range=[{corr.min():.2f}, {corr.max():.2f}]   <- note: NOT ~1.0")
print(f"  most coupled: {corr.idxmax()} ({corr.max():.2f});  "
      f"most buffered: {corr.idxmin()} ({corr.min():.2f})")

RNA-protein correlation: median=0.48, range=[-0.23, 0.92]   <- note: NOT ~1.0
  most coupled: VWA5A (0.92);  most buffered: ARPC1A (-0.23)


**Read this result.** The median correlation is well below 1 — most genes' protein levels only partly follow their RNA. That is the empirical reason you can't treat RNA as a stand-in for protein, and why integrating both layers adds information.

### 1.5 Per-layer differential signal
Each layer independently nominates candidates. Here the signal is the tumor-vs-normal effect size at RNA and protein, plus per-gene mutation frequency for the genomic layer.

In [7]:
if meta is not None:
    tcols = meta.index[meta == "tumor"]
    ncols = meta.index[meta == "normal"]
    rna_eff = (rna[tcols].mean(axis=1) - rna[ncols].mean(axis=1)).abs()
    prot_eff = (prot[tcols].mean(axis=1) - prot[ncols].mean(axis=1)).abs()
else:
    rna_eff = rna[samples].mean(axis=1).abs()
    prot_eff = prot[samples].mean(axis=1).abs()

scored = pd.DataFrame({"transcriptomic": rna_eff, "proteomic": prot_eff,
                       "genomic": mut_freq, "rna_prot_corr": corr})
scored.round(3).head()

,transcriptomic,proteomic,genomic,rna_prot_corr
A2M,0.060,0.494,0.033,0.349
A2ML1,0.954,3.147,0.016,NaN
AADACL2,0.499,0.396,0.008,NaN
AAED1,0.070,0.434,0.016,NaN
AAGAB,0.026,0.211,0.008,0.666


### 1.6 Multi-evidence score → ranked targets
Normalize each layer to a common scale and take the **equal-weighted** sum. This turns three noisy layers into one ranked list — the artifact everything downstream consumes.

In [8]:
scored["score"] = multi_evidence_score(
    scored, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
ranked_cptac = scored.sort_values("score", ascending=False)
print("Top CPTAC targets (worked example):")
print(ranked_cptac.head(6).round(3).to_string())
# Continuity check: TP53 / p53 (Module 1's P04637 / 1TUP) is a top breast-cancer hit.
print("\nTP53 rank:", list(ranked_cptac.index).index("TP53") + 1)

Top CPTAC targets (worked example):
         transcriptomic  proteomic  genomic  rna_prot_corr  score
SI                0.977      5.200    0.049            NaN  0.988
VWDE              1.263      1.424    0.049            NaN  0.979
MUC5B             0.566      3.441    0.082            NaN  0.978
SPHKAP            0.573      3.321    0.041            NaN  0.970
CEACAM5           0.813      3.315    0.033            NaN  0.970
RIMS2             1.240      1.676    0.033            NaN  0.968

TP53 rank: 108


### Part 1 recap
You integrated **at the sample level** because the data was **matched** — you could correlate RNA and protein within the same tumors. That was synthetic data, so the method is easy to see. Next we run it on real data, then you transfer it to unmatched data yourself.

---
# Part 2 — Find and load real CPTAC data *(together, in class)*

Now swap the synthetic files for the real thing. CPTAC breast data is open access (no data-use agreement) — we'll get it together in class. **Not graded.**

1. Go to **LinkedOmics** (`linkedomics.org`) → **CPTAC Breast Cancer (BRCA)**.
2. Download the gene-level **RNAseq** and **Proteome** matrices (gene × sample).
3. Save them into `data/` with genes as the row index and these exact names — transpose with `.T` first if samples are in rows:
   - `data/cptac_brca_rna.tsv`
   - `data/cptac_brca_protein.tsv`
   - `data/cptac_brca_mutation.tsv`  (one row per gene: mutation frequency or CNV magnitude)
4. `Kernel → Restart & Run All` and watch **Part 1** again — it now loads your real files instead of the demo. Compare: is the RNA–protein correlation still modest? Does TP53 still surface near the top?

*Wrinkle:* the real download has no tumor/normal labels, so ranking falls back to overall abundance rather than a tumor-vs-normal effect. That's fine for seeing the pipeline run on real data.

---
# Part 3 — Your assignment: Alzheimer's disease *(complete the `# TODO` cells)*

The three matrices you downloaded from Canvas are **gene-level summaries** from different cohorts — there are no shared samples to correlate, so you integrate **at the gene level.** You'll transfer the Part 1 workflow: harmonize → concordance → score → rank → export.

Your ranked target list (the CSV you export in 3.5) is what **Week 3** builds on, so it has to be clean. *Continuity:* expect **APOE** (`rs7412`) among your top hits.

### 3.1 — TODO: load and inspect the three matrices
Call `load_ad()` and look at each table's columns and shape before you touch them.

In [9]:
import pandas as pd, numpy as np, glob

hits = sorted(glob.glob("data/**/gwas*.tsv", recursive=True))
print("Files found:")
for h in hits:
    print("   ", h)

raw = pd.read_csv(hits[-1], sep="\t", low_memory=False)
print("\nUsing:", hits[-1], "| shape:", raw.shape)
print("\nColumns:")
for c in raw.columns:
    print("   ", c)
print("\nTraits present:")
print(raw["DISEASE/TRAIT"].value_counts().head(10).to_string())

Files found:
    data\AD\gwas-association-downloaded_2026-09-23-EFO_0000249.tsv
    data\AD\gwas-association-downloaded_2026-09-23-MONDO_0004975-withChildTraits.tsv

Using: data\AD\gwas-association-downloaded_2026-09-23-MONDO_0004975-withChildTraits.tsv | shape: (7410, 38)

Columns:
    DATE ADDED TO CATALOG
    PUBMEDID
    FIRST AUTHOR
    DATE
    JOURNAL
    LINK
    STUDY
    DISEASE/TRAIT
    INITIAL SAMPLE SIZE
    REPLICATION SAMPLE SIZE
    REGION
    CHR_ID
    CHR_POS
    REPORTED GENE(S)
    MAPPED_GENE
    UPSTREAM_GENE_ID
    DOWNSTREAM_GENE_ID
    SNP_GENE_IDS
    UPSTREAM_GENE_DISTANCE
    DOWNSTREAM_GENE_DISTANCE
    STRONGEST SNP-RISK ALLELE
    SNPS
    MERGED
    SNP_ID_CURRENT
    CONTEXT
    INTERGENIC
    RISK ALLELE FREQUENCY
    P-VALUE
    PVALUE_MLOG
    P-VALUE (TEXT)
    OR or BETA
    95% CI (TEXT)
    PLATFORM [SNPS PASSING QC]
    CNV
    MAPPED_TRAIT
    MAPPED_TRAIT_URI
    STUDY ACCESSION
    GENOTYPING TECHNOLOGY

Traits present:
DISEASE/TRAIT
Alzhei

In [10]:
import pandas as pd, numpy as np, glob

path = glob.glob("data/**/gwas-association-downloaded*MONDO_0004975*.tsv", recursive=True)[0]
raw = pd.read_csv(path, sep="\t", low_memory=False)
print("Using:", path)
print("Total associations:", len(raw))
print("\nAll traits:")
print(raw["DISEASE/TRAIT"].value_counts().to_string())

Using: data\AD\gwas-association-downloaded_2026-09-23-MONDO_0004975-withChildTraits.tsv
Total associations: 7410

All traits:
DISEASE/TRAIT
Alzheimer's disease or family history of Alzheimer's disease                                                                                         2264
Alzheimer's disease                                                                                                                                  1253
Alzheimer’s disease polygenic risk score (upper quantile vs lower quantile)                                                                           477
Alzheimer's disease, proxy Alzheimer's disease or related dementias                                                                                   383
Alzheimer's disease (adjusted for APOE e4 dosage)                                                                                                     268
Late-onset Alzheimer's disease                                                            

In [11]:
drop_terms = ["age of onset", "polygenic risk score", "rate of cognitive decline",
              "gastroesophageal", "progression", "biomarker", "neuroimaging",
              "cerebrospinal", "adjusted for APOE"]

mask = raw["DISEASE/TRAIT"].str.lower().str.contains("|".join(drop_terms), na=False)
print("Dropping", mask.sum(), "associations from non-risk traits")
print(raw.loc[mask, "DISEASE/TRAIT"].value_counts().to_string())

kept = raw[~mask].copy()
print("\nKeeping", len(kept), "associations across", kept["DISEASE/TRAIT"].nunique(), "traits")

kept["neglog10p"] = pd.to_numeric(kept["PVALUE_MLOG"], errors="coerce")

g = kept[["MAPPED_GENE", "neglog10p"]].dropna()
g["MAPPED_GENE"] = g["MAPPED_GENE"].astype(str).str.replace(" - ", ", ", regex=False)
g = g.assign(gene=g["MAPPED_GENE"].str.split(",")).explode("gene")
g["gene"] = g["gene"].str.strip().str.upper()
g = g[(g["gene"] != "") & (g["gene"] != "NAN")]

ad_gwas = g.groupby("gene", as_index=False)["neglog10p"].max()
ad_gwas.to_csv("data/ad_gwas.tsv", sep="\t", index=False)

print("\nWrote data/ad_gwas.tsv —", len(ad_gwas), "genes")
print(ad_gwas.sort_values("neglog10p", ascending=False).head(12).to_string(index=False))

Dropping 1344 associations from non-risk traits
DISEASE/TRAIT
Alzheimer’s disease polygenic risk score (upper quantile vs lower quantile)                477
Alzheimer disease and age of onset                                                         203
Alzheimer's disease or gastroesophageal reflux disease                                     186
Rate of cognitive decline in Alzheimer's disease                                           184
Alzheimer's disease, proxy Alzheimer's disease or related dementias (age of onset <75)     100
Alzheimer's disease, proxy Alzheimer's disease or related dementias (age of onset >=75)     97
Alzheimer's disease biomarkers                                                              26
Alzheimer's disease (age of onset)                                                          18
Alzheimer's disease or gastroesophageal reflux disease and/or peptic ulcer disease          14
Alzheimer's disease (age of onset) in APOE e4 non-carriers                         

In [12]:
import pandas as pd, glob

cands = [c for c in glob.glob("data/**/*67333*", recursive=True)
         if "design" not in c.lower()]

if not cands:
    print("Analytics file not downloaded yet. Files I can see:")
    for c in glob.glob("data/**/*67333*", recursive=True):
        print("   ", c)
else:
    path = cands[0]
    tx_raw = pd.read_csv(path, sep="\t", comment="#")
    print("Using:", path)
    print("Shape:", tx_raw.shape)
    print("Columns:", list(tx_raw.columns))
    print(tx_raw.head(3).to_string())

Analytics file not downloaded yet. Files I can see:
    data\AD\E-GEOD-67333-experiment-design.tsv


In [13]:
import glob
print("Everything in data/:")
for f in sorted(glob.glob("data/**/*", recursive=True)):
    print("   ", f)

Everything in data/:
    data\AD
    data\AD\13024_2021_474_MOESM1_ESM.xlsx
    data\AD\E-GEOD-67333-experiment-design.tsv
    data\AD\GSE5281.top.table.tsv
    data\AD\ad_gwas.tsv
    data\AD\gwas-association-downloaded_2026-09-23-EFO_0000249.tsv
    data\AD\gwas-association-downloaded_2026-09-23-MONDO_0004975-withChildTraits.tsv
    data\ad_gwas.tsv
    data\ad_proteomics.tsv
    data\ad_transcriptomics.tsv
    data\cptac_brca_mutation.tsv
    data\cptac_brca_protein.tsv
    data\cptac_brca_rna.tsv


In [14]:
import pandas as pd, numpy as np, os, shutil, glob

# --- make sure ad_gwas.tsv is in data/, not data/AD/ ---
if not os.path.exists("data/ad_gwas.tsv"):
    found = glob.glob("data/**/ad_gwas.tsv", recursive=True)
    if found:
        shutil.copy(found[0], "data/ad_gwas.tsv")
        print("moved ad_gwas.tsv ->", "data/ad_gwas.tsv")
print("ad_gwas.tsv in data/:", os.path.exists("data/ad_gwas.tsv"))

# --- load the GEO2R table ---
path = glob.glob("data/**/GSE5281.top.table.tsv", recursive=True)[0]
tx_raw = pd.read_csv(path, sep="\t")
print("\nLoaded:", path)
print("Rows (probes):", len(tx_raw))
print("Columns:", list(tx_raw.columns))

ad_gwas.tsv in data/: True

Loaded: data\AD\GSE5281.top.table.tsv
Rows (probes): 54675
Columns: ['ID', 'adj.P.Val', 'P.Value', 't', 'B', 'logFC', 'Gene.symbol', 'Gene.title']


In [15]:
tx = tx_raw[["Gene.symbol", "logFC", "P.Value"]].copy()
tx = tx.dropna(subset=["Gene.symbol", "logFC", "P.Value"])

# Affymetrix probes can map to several genes: "HBA2///HBA1"
tx = tx.assign(gene=tx["Gene.symbol"].astype(str).str.split("///")).explode("gene")
tx["gene"] = tx["gene"].str.strip().str.upper()
tx = tx[tx["gene"] != ""]

# sex-linked genes — the XIST problem you spotted
SEX_GENES = ["XIST","TSIX","RPS4Y1","RPS4Y2","DDX3Y","KDM5D","UTY",
             "USP9Y","EIF1AY","NLGN4Y","ZFY","TXLNGY","PRKY"]
hit = tx[tx["gene"].isin(SEX_GENES)]
print("Sex-linked rows removed:", len(hit), "| genes:", sorted(hit["gene"].unique()))
tx = tx[~tx["gene"].isin(SEX_GENES)]

# several probes per gene -> keep the one with the strongest p-value
tx = tx.sort_values("P.Value").drop_duplicates("gene", keep="first")

tx = tx.rename(columns={"logFC": "log2fc", "P.Value": "pval"})[["gene","log2fc","pval"]]
tx.to_csv("data/ad_transcriptomics.tsv", sep="\t", index=False)

print("\nWrote data/ad_transcriptomics.tsv —", len(tx), "genes")
print(tx.head(15).to_string(index=False))

known = ["APP","MAPT","APOE","CLU","BIN1","PICALM","TREM2","PSEN1","SNCA","GFAP"]
print("\nKnown AD genes present:")
print(tx[tx["gene"].isin(known)].to_string(index=False))

Sex-linked rows removed: 34 | genes: ['DDX3Y', 'EIF1AY', 'KDM5D', 'NLGN4Y', 'PRKY', 'RPS4Y1', 'TSIX', 'TXLNGY', 'USP9Y', 'UTY', 'XIST', 'ZFY']

Wrote data/ad_transcriptomics.tsv — 22822 genes
       gene    log2fc     pval
      IFI16 -6.547862 0.000410
       HBA2  6.504311 0.000448
       HBA1  6.504311 0.000448
      HAUS2 -6.369854 0.000587
       ZEB2 -6.042692 0.001110
 ATP2A1-AS1  5.981158 0.001250
ANKRD10-IT1 -5.894957 0.001470
      GRIN1  5.807532 0.001720
      LCORL -5.805631 0.001730
  AFAP1-AS1  5.805494 0.001730
   SLC25A37 -5.642919 0.002320
       EOGT -5.616674 0.002440
     PTPN18 -5.593638 0.002540
       COPA -5.550185 0.002740
      TSHZ2  5.505469 0.002970

Known AD genes present:
  gene    log2fc  pval
PICALM  2.904028 0.117
  MAPT  2.167649 0.242
  SNCA  2.131786 0.250
 PSEN1  2.091165 0.259
 TREM2 -1.746569 0.346
  GFAP -1.479973 0.424
  APOE  1.306116 0.481
   CLU  0.969677 0.601
  BIN1 -0.744886 0.688
   APP -0.609034 0.742


In [16]:
import pandas as pd, glob

f = glob.glob("data/ad/*.xlsx") + glob.glob("data/ad/*.xls")
print("Found:", f)

xl = pd.ExcelFile(f[0])
print("\nSheets:", xl.sheet_names)

for s in xl.sheet_names:
    d = pd.read_excel(f[0], sheet_name=s, nrows=5)
    print(f"\n--- {s} --- shape guess: {d.shape[1]} cols")
    print("Columns:", list(d.columns))

Found: ['data/ad\\13024_2021_474_MOESM1_ESM.xlsx']

Sheets: ['S1', 'S2', 'S3', 'S4', 'S5']

--- S1 --- shape guess: 5 cols
Columns: ['Unnamed: 0', 'Supplementary Table S1. The list of AD genes and risk loci with corresponding protein expressions in AD brains', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']

--- S2 --- shape guess: 28 cols
Columns: ['Unnamed: 0', 'Supplementary Table S2. Meta-analysis of seven published TMT datasets to identify proteomic changes in AD brains ', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27']

--- S3 --- shape guess: 12 cols
Columns: ['Unnamed: 0', 'Supplementary Table S3. Enrichment of AD differentially expressed proteins in diffe

In [17]:
import sys
!{sys.executable} -m pip install openpyxl

In [18]:
import pandas as pd, glob

f = glob.glob("data/ad/*.xlsx")[0]
xl = pd.ExcelFile(f)
print("Sheets:", xl.sheet_names)

for s in xl.sheet_names:
    d = pd.read_excel(f, sheet_name=s, header=None, nrows=4)
    print(f"\n=== {s} ===")
    print(d.to_string())

Sheets: ['S1', 'S2', 'S3', 'S4', 'S5']

=== S1 ===
    0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   1           2                                     3       4
0 NaN                                                                                                                                                                                                                                                                                                                                                            

In [19]:
import pandas as pd, numpy as np, glob, os

f = glob.glob("data/**/13024_2021_474_MOESM1_ESM.xlsx", recursive=True)[0]
raw = pd.read_excel(f, sheet_name="S2", header=None)

# find the row where column 1 says "Gene Name"
hdr = raw.index[raw[1].astype(str).str.strip() == "Gene Name"][0]
print("Header row found at index:", hdr)

data = raw.iloc[hdr+2:].copy()          # +2 skips the sub-header row
pr = pd.DataFrame({
    "gene":   data[1].astype(str).str.strip().str.upper(),
    "log2fc": pd.to_numeric(data[21], errors="coerce"),   # Median of log2(AD/ctl)
    "pval":   pd.to_numeric(data[20], errors="coerce"),   # Fisher's combined P
    "fdr":    pd.to_numeric(data[22], errors="coerce"),   # BH FDR
})

pr = pr[(pr["gene"] != "") & (pr["gene"] != "NAN")]
pr = pr.dropna(subset=["log2fc", "pval"])
pr = pr.sort_values("pval").drop_duplicates("gene", keep="first")

print("Proteins:", len(pr), "| expected ~12,017")
print("Significant at FDR<0.01:", (pr["fdr"] < 0.01).sum(), "| paper reports 2,698")

pr[["gene","log2fc","pval"]].to_csv("data/ad_proteomics.tsv", sep="\t", index=False)
print("\nWrote data/ad_proteomics.tsv")

known = ["APP","MAPT","APOE","CLU","SMOC1","MDK","VGF","NPTX2","GFAP","TREM2","BIN1"]
print("\nKnown AD proteins:")
print(pr[pr["gene"].isin(known)][["gene","log2fc","pval","fdr"]].to_string(index=False))

Header row found at index: 2
Proteins: 12017 | expected ~12,017
Significant at FDR<0.01: 2698 | paper reports 2,698

Wrote data/ad_proteomics.tsv

Known AD proteins:
 gene    log2fc         pval          fdr
  MDK  2.563709 2.170162e-35 2.607884e-31
SMOC1  1.196933 6.468056e-35 3.886331e-31
NPTX2 -1.032371 3.668174e-26 4.408045e-22
  VGF -0.921850 4.967772e-25 2.984886e-21
 MAPT  0.482489 2.073371e-19 2.076309e-16
  CLU  0.441190 6.088234e-18 4.877487e-15
 GFAP  1.159123 2.167019e-17 1.302054e-14
 APOE  0.562984 4.098358e-15 1.172618e-12
  APP  0.074725 9.277771e-07 2.428997e-05
TREM2  0.347521 1.926029e-03 1.441858e-02
 BIN1 -0.041983 9.910752e-02 4.066149e-01


In [20]:
for fn in ["ad_transcriptomics.tsv", "ad_proteomics.tsv", "ad_gwas.tsv"]:
    print(f"data/{fn}:", os.path.exists(f"data/{fn}"))

data/ad_transcriptomics.tsv: True
data/ad_proteomics.tsv: True
data/ad_gwas.tsv: True


In [21]:
tx, pr, gw = load_ad()

In [22]:
tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx.merge(pr, on="gene").merge(gw, on="gene")
print("Genes surviving the 3-way join:", len(df))
print(df.head().to_string(index=False))

Genes surviving the 3-way join: 1317
    gene   rna_lfc   rna_p  prot_lfc   prot_p  neglog10p
HLA-DQB1  5.273416 0.00443  0.256569 0.170480  18.698970
  SPTBN1 -5.228384 0.00478 -0.031716 0.012348   5.045757
  MRPL39 -5.101673 0.00590 -0.061355 0.004320   6.522879
   NOC4L  4.895680 0.00824 -0.058702 0.021703   9.096910
   PRKD3 -4.881891 0.00842  0.141930 0.022301   9.000000


### 3.2 — TODO: harmonize on the gene symbol → one joined table
Rename the effect/`pval` columns so RNA and protein don't collide, then merge all three on `gene`. Report how many genes survive the join (your first reality check).

In [23]:
# TODO 3.2 — build a single joined table `df`.
# Hint: rename before merging so columns don't clash.
tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx.merge(pr, on="gene").merge(gw, on="gene")
print("Genes surviving the 3-way join:", len(df))

Genes surviving the 3-way join: 1317


### 3.3 — TODO: sign-agreement concordance
You can't correlate across samples here (nothing is matched), so concordance becomes **direction agreement**: does the gene move the *same way* at RNA and protein? Add a boolean `concordant` column. Watch for genes that are strong at RNA but flat/opposite at protein — those are the interesting discordant ones.

In [24]:
# TODO 3.3 — add a boolean `concordant` column: do RNA and protein point the same direction?
# Hint: compare np.sign(df["rna_lfc"]) with np.sign(df["prot_lfc"])
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print("Concordant:", df["concordant"].sum(), "of", len(df),
      f"({df['concordant'].mean():.1%})")

Concordant: 618 of 1317 (46.9%)


### 3.4 — TODO: multi-evidence score
Build a per-layer magnitude for each of the three layers, then score with `multi_evidence_score` and `EQUAL_WEIGHTS`. If you change the weights, justify it in your report.

In [25]:
# TODO 3.4 — score every gene.
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"]      = df["prot_lfc"].abs()
df["genomic"]        = df["neglog10p"]

df["score"] = multi_evidence_score(df, ["transcriptomic","proteomic","genomic"], EQUAL_WEIGHTS)

### 3.5 — TODO: rank, inspect, and export
Sort by score, take the top ~15, and **export the ranked table to `targets_ad.csv`** — this file is the hand-off to Week 3. Check whether known AD genes (APOE, TREM2, BIN1, CLU, PICALM …) are recovered, and look at any `concordant == False` genes in your top hits.

In [26]:
# TODO 3.5 — rank, look at the top 15, and export the CSV.
ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

cols = ["gene","rna_lfc","rna_p","prot_lfc","prot_p","neglog10p","concordant","score"]
top15 = ranked[cols].head(15)
print(top15.to_string(index=False))

known = ["APOE","TREM2","BIN1","CLU","PICALM","APP","MAPT","CR1","ABCA7","SORL1","PLCG2","INPP5D"]
print("\nKnown AD genes in the full ranking:")
for g in known:
    if g in ranked["gene"].values:
        print(f"   {g}: rank {ranked.index[ranked['gene']==g][0]+1} of {len(ranked)}")
    else:
        print(f"   {g}: not in joined set")

print("\nDiscordant genes in top 15:")
print(top15[~top15["concordant"]][["gene","rna_lfc","prot_lfc","neglog10p"]].to_string(index=False))

ranked[cols].head(15).to_csv("targets_ad.csv", index=False)
print("\nWrote targets_ad.csv")

    gene   rna_lfc   rna_p  prot_lfc       prot_p  neglog10p  concordant    score
HLA-DQB1  5.273416 0.00443  0.256569 1.704796e-01  18.698970        True 0.952670
   CD2AP -3.380683 0.06810  0.222863 1.924951e-08  21.096910       False 0.933435
HLA-DQA1  3.543945 0.05580  0.191261 1.650182e-01  22.522879        True 0.930018
  PECAM1 -3.306959 0.07430  0.246992 2.221043e-06  16.522879       False 0.924956
   CABP1  3.522583 0.05730 -0.256132 2.800862e-09  14.698970       False 0.924829
   ABCA1 -3.830654 0.03870  0.150932 5.768202e-03  32.045757       False 0.914452
  SAMHD1  2.434545 0.18900  0.249906 2.247514e-08  35.698970        True 0.909896
ARHGAP15  3.804391 0.04010  0.302475 3.946480e-02  11.000000        True 0.903948
   NUMA1  2.892938 0.11800  0.241269 2.754246e-05  14.000000        True 0.893065
   PILRA  2.149308 0.24600  0.282430 2.096007e-02  23.397940        True 0.889269
     CR1  2.091010 0.25900  0.270252 1.494293e-01  55.154902        True 0.889142
  CLPTM1  3.0620

In [27]:
for g in ["APOE","APOC1","TOMM40","BIN1","CLU","PICALM","TREM2","APP"]:
    if g in ranked["gene"].values:
        r = ranked[ranked["gene"]==g].iloc[0]
        print(f"{g:8s} rank {ranked.index[ranked['gene']==g][0]+1:5d}  "
              f"rna={r['rna_lfc']:7.2f}  prot={r['prot_lfc']:7.3f}  gwas={r['neglog10p']:7.2f}  score={r['score']:.3f}")
    else:
        print(f"{g:8s} not in joined set")

APOE     rank    34  rna=   1.31  prot=  0.563  gwas= 323.00  score=0.821
APOC1    rank   220  rna=   0.75  prot= -0.140  gwas= 307.70  score=0.665
TOMM40   rank   196  rna=   1.43  prot= -0.072  gwas= 295.00  score=0.680
BIN1     rank   609  rna=  -0.74  prot= -0.042  gwas= 138.70  score=0.520
CLU      rank    82  rna=   0.97  prot=  0.441  gwas=  47.00  score=0.763
PICALM   rank    50  rna=   2.90  prot= -0.071  gwas=  50.15  score=0.800
TREM2    rank    20  rna=  -1.75  prot=  0.348  gwas=  37.52  score=0.868
APP      rank   595  rna=  -0.61  prot=  0.075  gwas=  13.70  score=0.524


In [28]:
readme = """# BIOT 6900 — Module 2, Part 3
Krupa Patel

## Data sources

**Genomics — GWAS Catalog**
- Trait: Alzheimer disease (MONDO_0004975, with child traits)
- URL: https://www.ebi.ac.uk/gwas/
- Downloaded: 2026-09-23 | Access: Open
- 7,418 associations; filtered out non-risk traits (age of onset, PRS,
  cognitive decline, APOE-adjusted, multi-trait); collapsed to 2,970 genes
  keeping the strongest variant per gene

**Transcriptomics — GEO GSE5281 via GEO2R**
- Brain region: FILL IN | AD samples: FILL IN | Controls: FILL IN
- URL: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE5281
- Access: Open
- Collapsed probes to genes keeping the strongest p-value per gene;
  removed 34 sex-linked rows after XIST dominated the initial comparison
- 22,822 genes

**Proteomics — Bai et al. 2021**
- Mol Neurodegener 16:55, Supplementary Table S2
- DOI: 10.1186/s13024-021-00474-z | Access: Open
- Meta-analysis of 7 TMT datasets, 192 AD and control cortical specimens
- 12,017 proteins

## Results
3-way join: 1,317 genes
Concordance: 618/1,317 (46.9%)
Output: targets_ad.csv (top 15 ranked targets)

## Notes
FILL IN — anything that didn't work or needed a workaround
"""

with open("README.md", "w", encoding="utf-8") as fh:
    fh.write(readme)

print("Wrote README.md")
print(open("README.md").read()[:400])

Wrote README.md
# BIOT 6900 â€” Module 2, Part 3
Krupa Patel

## Data sources

**Genomics â€” GWAS Catalog**
- Trait: Alzheimer disease (MONDO_0004975, with child traits)
- URL: https://www.ebi.ac.uk/gwas/
- Downloaded: 2026-09-23 | Access: Open
- 7,418 associations; filtered out non-risk traits (age of onset, PRS,
  cognitive decline, APOE-adjusted, multi-trait); collapsed to 2,970 genes
  keeping the strongest 


### 3.6 — Interpretation (write-up, goes in your report)
Answer these in your 3–4 page report — this is the 40% interpretation payload:

1. **Weighting.** You used equal weights. Argue for keeping them equal *or* for up-weighting a layer (e.g. GWAS as germline/causal-leaning vs. transcriptomics as possibly downstream). There's no single right answer — only reasoned vs. unreasoned.
2. **Top targets.** Which known AD genes did you recover (APOE, TREM2, BIN1, CLU, PICALM …)? Any non-obvious hit worth a second look?
3. **Read a discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA/GWAS, flat protein). What biology could explain RNA and protein disagreeing?
4. **Limitation.** This was **gene-level, cross-cohort** integration — unmatched — so you *cannot* make per-patient claims. Contrast this with the matched CPTAC case from Part 1.

---
### Submit (Part 3 only)
1. `Kernel → Restart & Run All` — confirm the whole notebook runs top to bottom.
2. Commit **this notebook**, **`targets_ad.csv`**, and a short **README** (your name + anything that didn't work) to your `biot6900` repo.
3. Push, confirm the files appear on github.com, then **post your repo link on Canvas.**

Grading follows the course 60 / 40 split — 60 execution, 40 interpretation & communication. Partial credit for a correct approach even with minor technical errors: if a step wouldn't run, say what you were trying to do and what happened.